# Drug-Drug Interaction (DDI) Detection
**Department of Pharmacy — TMU**  
**Dataset:** Patient Medication Records — DDI Case Group 6  
**Task:** Binary Classification — Drug Interaction Found (Yes/No)  
**Model:** Logistic Regression (class_weight='balanced')

---

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)

print('Libraries loaded.')

## Step 2 — Load Dataset

In [ ]:
df_raw = pd.read_excel('DDI CASE EXCEL GROUP 6.xlsx', sheet_name='Form Responses 1', header=0)

print(f'Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')

## Step 3 — Data Overview

In [ ]:
print('=== Target Distribution (Drug Interaction Found) ===')
print(df_raw['DRUG INTERACTION FOUND'].value_counts(dropna=False))

print('\n=== Interaction Types ===')
print(df_raw['TYPES OF INTERACTION '].value_counts(dropna=False))

print('\n=== Severity Distribution ===')
print(df_raw['SEVERITY'].value_counts(dropna=False))

print('\n=== Null Counts ===')
print(df_raw.isnull().sum())

## Step 4 — Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# --- Target ---
df['TARGET'] = df['DRUG INTERACTION FOUND'].str.strip().str.upper()
df = df[df['TARGET'].isin(['YES', 'NO'])].copy()
df['TARGET_BINARY'] = (df['TARGET'] == 'YES').astype(int)

# --- Age ---
df['AGE_CLEAN'] = df['AGE'].astype(str).str.extract(r'(\d+)').astype(float)

# --- Gender ---
df['GENDER_BINARY'] = df['GENDER'].str.strip().str.upper().apply(
    lambda x: 1 if x == 'MALE' else 0
)

# --- Comorbidities ---
df['HAS_COMORBIDITY'] = df['COMORBIDITIES'].apply(
    lambda x: 0 if (pd.isna(x) or str(x).strip().upper() == 'NO') else 1
)

# --- Severity: ordinal encoding ---
severity_map = {'NO': 0, 'MILD': 1, 'MONITOR CLOSELY': 2, 'MODERATE': 3, 'SEVERE': 4}
df['SEVERITY_ENCODED'] = df['SEVERITY'].str.strip().str.upper().map(severity_map).fillna(0)

# --- Interaction type flags ---
df['IS_PHARMACODYNAMIC'] = df['TYPES OF INTERACTION '].str.strip().str.upper().str.contains('PHARMACODYNAMIC', na=False).astype(int)
df['IS_PHARMACOKINETIC'] = df['TYPES OF INTERACTION '].str.strip().str.upper().str.contains('PHARMACOKINETIC', na=False).astype(int)

# --- Mechanism flags ---
df['MECH_CYP']       = df['MECHANISM'].astype(str).str.upper().str.contains('CYP',              na=False).astype(int)
df['MECH_ADDITIVE']  = df['MECHANISM'].astype(str).str.upper().str.contains('ADDITIVE|ANTAGONIST', na=False).astype(int)
df['MECH_SYNERGISM'] = df['MECHANISM'].astype(str).str.upper().str.contains('SYNERG',           na=False).astype(int)
df['MECH_QT']        = df['MECHANISM'].astype(str).str.upper().str.contains('QT',               na=False).astype(int)

print('Cleaning complete.')
print(f'Records: {len(df)}')
print(f'Target distribution: {df["TARGET_BINARY"].value_counts().to_dict()}')

## Step 5 — Feature Selection & Train/Test Split

In [ ]:
FEATURES = [
    'AGE_CLEAN',
    'GENDER_BINARY',
    'HAS_COMORBIDITY',
    'SEVERITY_ENCODED',
    'IS_PHARMACODYNAMIC',
    'IS_PHARMACOKINETIC',
    'MECH_CYP',
    'MECH_ADDITIVE',
    'MECH_SYNERGISM',
    'MECH_QT'
]

df_model = df[FEATURES + ['TARGET_BINARY']].dropna()
X = df_model[FEATURES]
y = df_model['TARGET_BINARY']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train distribution: {y_train.value_counts().to_dict()}')
print(f'Test  distribution: {y_test.value_counts().to_dict()}')

## Step 6 — Train Model

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)
model.fit(X_train_sc, y_train)
print('Model training complete.')

## Step 7 — Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print('=' * 50)
print('        CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['No DDI', 'DDI Found']))

print('=' * 50)
print('        KEY METRICS SUMMARY')
print('=' * 50)
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob):.4f}')

## Step 8 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: No DDI', 'Predicted: DDI Found'],
    yticklabels=['Actual: No DDI', 'Actual: DDI Found'],
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — DDI Detection', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — DDI Detection', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Odds Ratios

In [ ]:
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': model.coef_[0]
})
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])
coef_df = coef_df.sort_values('Odds_Ratio', ascending=True)

print('=== Odds Ratios per Feature ===')
print(coef_df[['Feature', 'Odds_Ratio']].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#d9534f' if v > 1 else '#5bc0de' for v in coef_df['Odds_Ratio']]
ax.barh(coef_df['Feature'], coef_df['Odds_Ratio'], color=colors, edgecolor='white')
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1, label='No effect (OR=1)')
ax.set_title('Odds Ratios — DDI Risk Factors', fontsize=14, fontweight='bold')
ax.set_xlabel('Odds Ratio', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('odds_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nInterpretation: Red = increased DDI risk | Blue = decreased DDI risk')

## Step 11 — Download Outputs

In [ ]:
from google.colab import files
files.download('confusion_matrix.png')
files.download('roc_curve.png')
files.download('odds_ratios.png')

---
## Summary

| Item | Detail |
|------|--------|
| Model | Logistic Regression |
| Imbalance Handling | class_weight='balanced' |
| Split | 80% train / 20% test (stratified) |
| Dataset Size | 278 patient records |
| Class Balance | 191 DDI-positive (69%) / 87 DDI-negative (31%) |

> **Note:** Unlike the ADR datasets, this dataset is reasonably balanced (69/31 split), which enables reliable classification performance. Interaction type and severity are the strongest predictors of DDI presence.